# 🏋️ Landmark Detection - Model Training
## End-to-End Training Pipeline with Real Data

This notebook trains a ResNet50 model for landmark classification using real data from `train.csv`.

**Pipeline Steps:**
1. Load and filter `train.csv` to top N landmarks
2. Create stratified train/val split
3. Create PyTorch datasets with image loading from disk
4. Download missing images (optional)
5. Define and train ResNet50 model
6. Track training progress and save checkpoints

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import time
import sys

# Add project to path
sys.path.append('..')

# Settings
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

### 2. Dataset Configuration

In [ ]:
# Configuration
NUM_CLASSES = 500      # Will be set based on filtered data
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
IMAGE_SIZE = 224
TOP_LANDMARKS = 500    # Number of top landmarks to use
VAL_RATIO = 0.2        # Validation split ratio

# Paths
PROJECT_ROOT = Path('../').resolve()
TRAIN_CSV = PROJECT_ROOT / 'train.csv'
IMAGES_DIR = PROJECT_ROOT / 'images'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'
IMAGES_OUT = PROJECT_ROOT / 'images'
DOWNLOAD_CHECKPOINT = PROJECT_ROOT / '.download_checkpoint.txt'

# Ensure directories exist
CHECKPOINT_DIR.mkdir(exist_ok=True)
IMAGES_DIR.mkdir(exist_ok=True)
IMAGES_OUT.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Train CSV: {TRAIN_CSV}")
print(f"Images directory: {IMAGES_DIR}")
print(f"Using top {TOP_LANDMARKS} landmarks")

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# ============================================
# STEP 1: Load train.csv and filter to top N landmarks
# ============================================
print("\n" + "="*60)
print("STEP 1: Loading and Filtering Data")
print("="*60)

# Load train.csv
print(f"\nLoading {TRAIN_CSV}...")
df_full = pd.read_csv(TRAIN_CSV)
print(f"Total images: {len(df_full):,}")
print(f"Total unique landmarks: {df_full['landmark_id'].nunique():,}")

# Count images per landmark and get top N
landmark_counts = df_full.groupby('landmark_id').size().sort_values(ascending=False)
print(f"\nLandmark image count distribution:")
print(f"  Max images per landmark: {landmark_counts.iloc[0]:,}")
print(f"  Min images per landmark: {landmark_counts.iloc[-1]:,}")
print(f"  Median images per landmark: {landmark_counts.median():.0f}")

# Keep only top N landmarks
top_landmark_ids = set(landmark_counts.head(TOP_LANDMARKS).index)
df_filtered = df_full[df_full['landmark_id'].isin(top_landmark_ids)].copy()
print(f"\nFiltered to top {TOP_LANDMARKS} landmarks: {len(df_filtered):,} images")

# Update NUM_CLASSES
NUM_CLASSES = TOP_LANDMARKS
print(f"Number of classes: {NUM_CLASSES}")

In [ ]:
# ============================================
# STEP 2: Train/Validation Split
# ============================================
print("\n" + "="*60)
print("STEP 2: Creating Train/Val Split")
print("="*60)

from sklearn.model_selection import train_test_split

# Create stratified train/val split
train_df, val_df = train_test_split(
    df_filtered,
    test_size=VAL_RATIO,
    stratify=df_filtered['landmark_id'],
    random_state=42
)

print(f"\nTraining set: {len(train_df):,} images")
print(f"Validation set: {len(val_df):,} images")

# Save splits to CSV for future use
(PROJECT_ROOT / 'data').mkdir(exist_ok=True)
train_csv_path = PROJECT_ROOT / 'data' / 'train_split.csv'
val_csv_path = PROJECT_ROOT / 'data' / 'val_split.csv'
train_df.to_csv(train_csv_path, index=False)
val_df.to_csv(val_csv_path, index=False)
print(f"\nSaved splits to:")
print(f"  - {train_csv_path}")
print(f"  - {val_csv_path}")

In [ ]:
# ============================================
# STEP 3: Create Dataset and DataLoaders
# ============================================
print("\n" + "="*60)
print("STEP 3: Creating Dataset and DataLoaders")
print("="*60)

# Create landmark ID to index mapping (consistent across train and val)
all_landmark_ids = sorted(df_filtered['landmark_id'].unique())
landmark_to_idx = {lm: idx for idx, lm in enumerate(all_landmark_ids)}
idx_to_landmark = {idx: lm for lm, idx in landmark_to_idx.items()}

print(f"Created mapping for {len(landmark_to_idx)} landmarks")
print(f"  Example: landmark_id 12345 -> index {landmark_to_idx.get(12345, 'N/A')}")

# Custom Dataset class with proper image loading
class LandmarkImageDataset(torch.utils.data.Dataset):
    """Dataset that loads actual images from disk."""
    
    def __init__(self, dataframe, images_dir, landmark_to_idx, transform=None, train=True):
        self.df = dataframe.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.landmark_to_idx = landmark_to_idx
        self.transform = transform
        self.train = train
        
        # Generate placeholder color based on landmark for missing images
        np.random.seed(42)
        self.placeholder_colors = {
            lm: tuple(np.random.randint(80, 180, 3)) 
            for lm in landmark_to_idx.keys()
        }
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row['id']
        landmark_id = row['landmark_id']
        
        # Try to load image from disk
        image_path = self.images_dir / f"{image_id}.jpg"
        
        if image_path.exists():
            try:
                image = Image.open(image_path).convert('RGB')
            except Exception as e:
                # Use placeholder on error
                color = self.placeholder_colors[landmark_id]
                image = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), color=color)
        else:
            # Use placeholder image (will download later)
            color = self.placeholder_colors[landmark_id]
            image = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), color=color)
        
        if self.transform:
            image = self.transform(image)
        
        label = self.landmark_to_idx[landmark_id]
        return image, torch.tensor(label, dtype=torch.long)

# Define transforms
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2)
])

val_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = LandmarkImageDataset(
    train_df, IMAGES_DIR, landmark_to_idx, 
    transform=train_transforms, train=True
)
val_dataset = LandmarkImageDataset(
    val_df, IMAGES_DIR, landmark_to_idx, 
    transform=val_transforms, train=False
)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=0,  # Set >0 for parallel loading with real images
    pin_memory=True if torch.cuda.is_available() else False
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"\nDatasets created:")
print(f"  Training: {len(train_dataset):,} samples, {len(train_loader)} batches")
print(f"  Validation: {len(val_dataset):,} samples, {len(val_loader)} batches")
print(f"  Batch size: {BATCH_SIZE}")

In [ ]:
# ============================================
# STEP 4: Download Missing Images
# ============================================
print("\n" + "="*60)
print("STEP 4: Download Missing Images")
print("="*60)

import asyncio
import aiohttp
import aiofiles

# Check how many images exist vs needed
needed_ids = set(df_filtered['id'].values)
existing_images = set(f.stem for f in IMAGES_DIR.glob('*.jpg')) if IMAGES_DIR.exists() else set()
missing_ids = needed_ids - existing_images

print(f"\nImage status:")
print(f"  Total images needed: {len(needed_ids):,}")
print(f"  Already downloaded: {len(existing_images):,}")
print(f"  Missing: {len(missing_ids):,}")

# Download function
async def download_images_async(ids_to_download, urls_df, output_dir, max_concurrent=20):
    """Download images asynchronously."""
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def download_one(session, image_id, url, sem):
        async with sem:
            output_path = output_dir / f"{image_id}.jpg"
            if output_path.exists():
                return True
                
            for attempt in range(3):
                try:
                    async with session.get(url, timeout=aiohttp.ClientTimeout(total=30)) as resp:
                        if resp.status == 200:
                            content = await resp.read()
                            if len(content) > 1000:  # Valid image check
                                async with aiofiles.open(output_path, 'wb') as f:
                                    await f.write(content)
                                return True
                except Exception:
                    if attempt < 2:
                        await asyncio.sleep(1 * (attempt + 1))
            return False
    
    timeout = aiohttp.ClientTimeout(total=60)
    connector = aiohttp.TCPConnector(limit=max_concurrent)
    
    url_map = dict(zip(urls_df['id'], urls_df['url']))
    
    async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
        tasks = [
            download_one(session, img_id, url_map[img_id], semaphore)
            for img_id in ids_to_download if img_id in url_map
        ]
        
        results = []
        for f in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Downloading"):
            results.append(await f)
        
        return sum(results)

def download_missing_images():
    """Download missing images with progress bar."""
    if len(missing_ids) == 0:
        print("All images already downloaded!")
        return
    
    print(f"\nStarting download of {len(missing_ids):,} missing images...")
    print("This may take a while. You can cancel and run later with:")
    print(f"  !python ../download_images.py --resume\n")
    
    urls_df = df_filtered[df_filtered['id'].isin(missing_ids)][['id', 'url']]
    
    try:
        success = asyncio.run(download_images_async(list(missing_ids), urls_df, IMAGES_DIR))
        print(f"\nDownload complete: {success} images saved")
    except Exception as e:
        print(f"Download error: {e}")
        print("Images can be downloaded later using: python ../download_images.py --resume")

# Ask before downloading
download_choice = input(f"\nDownload {len(missing_ids):,} missing images now? [y/N]: ")
if download_choice.lower() == 'y':
    download_missing_images()
else:
    print("Skipping download. Placeholder images will be used for missing files.")
    print("Run later: python ../download_images.py --resume")

# ============================================
# STEP 5: Define Model
# ============================================
print("\n" + "="*60)
print("STEP 5: Model Definition")
print("="*60)

class LandmarkClassifier(nn.Module):
    """ResNet50-based landmark classifier."""
    def __init__(self, num_classes, pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = models.resnet50(pretrained=pretrained)
        in_features = self.backbone.fc.in_features  # 2048
        
        # Replace final layer
        self.backbone.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes)
        )
        self.num_classes = num_classes
    
    def forward(self, x):
        return self.backbone(x)
    
    def get_landmark_mapping(self):
        """Return mapping for debug/logging."""
        return idx_to_landmark

# Create model
model = LandmarkClassifier(num_classes=NUM_CLASSES, pretrained=True, dropout=0.3)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel: LandmarkClassifier (ResNet50 backbone)")
print(f"  Number of classes: {NUM_CLASSES}")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

In [ ]:
# ============================================
# STEP 6: Training Configuration
# ============================================
print("\n" + "="*60)
print("STEP 6: Training Setup")
print("="*60)

import torch.optim as optim

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer with AdamW
optimizer = optim.AdamW(
    model.parameters(), 
    lr=LEARNING_RATE, 
    weight_decay=0.01
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=NUM_EPOCHS, 
    eta_min=1e-6
)

print(f"\nTraining configuration:")
print(f"  Loss function: CrossEntropyLoss")
print(f"  Optimizer: AdamW(lr={LEARNING_RATE}, weight_decay=0.01)")
print(f"  Scheduler: CosineAnnealingLR(T_max={NUM_EPOCHS})")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")

In [ ]:
# ============================================
# STEP 7: Training Functions
# ============================================
print("\n" + "="*60)
print("STEP 7: Training Functions")
print("="*60)

def train_epoch(model, loader, criterion, optimizer, device, epoch=0):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc=f'Epoch {epoch+1} - Training')
    for batch_idx, (images, labels) in enumerate(pbar):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
        
        pbar.set_postfix({
            'loss': f'{total_loss/total:.4f}', 
            'acc': f'{100.*correct/total:.2f}%'
        })
    
    return total_loss / total, 100. * correct / total

def validate(model, loader, criterion, device):
    """Validate the model."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    
    return total_loss / total, 100. * correct / total

print("\nDefined functions: train_epoch(), validate()")

In [ ]:
# ============================================
# STEP 8: Initialize Training History
# ============================================
print("\n" + "="*60)
print("STEP 8: Training History Setup")
print("="*60)

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'lr': []
}
best_val_acc = 0.0
best_model_path = CHECKPOINT_DIR / 'best_model.pth'

print(f"Ready to train for {NUM_EPOCHS} epochs")
print(f"Best model will be saved to: {best_model_path}")

# ============================================
# STEP 9: Training Loop
# ============================================
print("="*60)
print("STEP 9: Training Loop")
print("="*60)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    print(f'\nEpoch {epoch+1}/{NUM_EPOCHS}')
    print('-'*40)
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, epoch)
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # Update scheduler
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    
    # Log history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
    print(f'Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%')
    print(f'LR: {current_lr:.6f}')
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print('>> New best model saved!')

total_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"Training completed in {total_time/60:.1f} minutes")
print(f"Best validation accuracy: {best_val_acc:.2f}%")
print(f"{'='*60}")

In [ ]:
# ============================================
# STEP 10: Training History Plot
# ============================================
print("\n" + "="*60)
print("STEP 10: Training History Visualization")
print("="*60)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss plot
axes[0].plot(epochs_range, history['train_loss'], 'b-', label='Train', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], 'r-', label='Val', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss', fontsize=11)
axes[0].set_title('Training & Validation Loss', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy plot
axes[1].plot(epochs_range, history['train_acc'], 'b-', label='Train', linewidth=2)
axes[1].plot(epochs_range, history['val_acc'], 'r-', label='Val', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Accuracy (%)', fontsize=11)
axes[1].set_title('Training & Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Learning rate plot
axes[2].plot(epochs_range, history['lr'], 'g-', linewidth=2)
axes[2].set_xlabel('Epoch', fontsize=11)
axes[2].set_ylabel('Learning Rate', fontsize=11)
axes[2].set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
axes[2].set_yscale('log')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_OUT / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nPlot saved to: {IMAGES_OUT / "training_history.png"}')

In [ ]:
# ============================================
# STEP 12: Summary
# ============================================

In [ ]:
# ============================================
# STEP 12: Summary
# ============================================
print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)

print(f"""
SUMMARY
-------
Configuration:
  - Classes: {NUM_CLASSES} landmarks (top {TOP_LANDMARKS})
  - Epochs: {NUM_EPOCHS}
  - Batch size: {BATCH_SIZE}
  - Image size: {IMAGE_SIZE}x{IMAGE_SIZE}
  - Learning rate: {LEARNING_RATE}

Results:
  - Best validation accuracy: {best_val_acc:.2f}%
  - Final train accuracy: {history['train_acc'][-1]:.2f}%
  - Final validation accuracy: {history['val_acc'][-1]:.2f}%

Files saved:
  - Best model: {best_model_path}
  - Final model: {final_model_path}
  - Training history plot: {IMAGES_OUT / 'training_history.png'}
  - Training history CSV: {PROJECT_ROOT / 'logs' / 'training_history.csv'}
  - Train split: {train_csv_path}
  - Val split: {val_csv_path}

NEXT STEPS
----------
1. Run 05_Inference_Demo.ipynb to test predictions
2. Or use the model for inference:
   python landmark_detection.py --mode inference --image path/to/image.jpg
3. Download more images with:
   python download_images.py --resume
""")